# Stage 2 — Feature Engineering

**Goal:** Transform raw columns into the exact numerical format ML models need,  
without introducing data leakage.

Key decisions we make here:
- **Binary columns** (Yes/No) → 0/1 directly
- **Nominal columns** (Contract, PaymentMethod) → OneHotEncoder, NOT LabelEncoder
- **Scaling** → fit StandardScaler on **train data only** — never on the full dataset
- **Class imbalance** → SMOTE on train set only
- **Derived features** → add domain knowledge as new columns

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from db import query
from features import add_derived_features, encode_binary, build_preprocessor, NOMINAL_COLS, NUMERIC_COLS

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded')

## 1. Load Customer 360

In [ ]:
df = query('SELECT * FROM analytics.customer_360')
print(f'Loaded {len(df):,} customers, {df.shape[1]} columns')
df.head(2)

## 2. Add Derived Features

These are features not in the raw data but meaningful for churn prediction.

| Feature | Logic | Business meaning |
|---|---|---|
| `tenure_band` | 0-12m=0, 13-36m=1, 37-60m=2, 61+m=3 | Customer lifecycle stage |
| `num_services` | count of active add-ons | Stickiness — more = less likely to churn |
| `charge_per_month_ratio` | monthly / total | High = new customer, hasn't invested much |
| `revenue_at_risk` | month-to-month AND charges > $70 | High-value customers with no lock-in |

In [ ]:
df = add_derived_features(df)

print('New features added:')
print(df[['tenure_band', 'num_services', 'charge_per_month_ratio', 'revenue_at_risk']].head(5))

In [ ]:
# Visualise derived features vs churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# num_services
ns_churn = df.groupby('num_services')['churn'].apply(lambda x: (x=='Yes').mean())
axes[0].bar(ns_churn.index, ns_churn.values, color='#3498db', edgecolor='white')
axes[0].set_title('Churn Rate by Number of Services', fontweight='bold')
axes[0].set_xlabel('Active Add-on Services')
axes[0].set_ylabel('Churn Rate')

# tenure_band
tb_labels = {0: 'New\n(0-12m)', 1: 'Growing\n(13-36m)', 2: 'Loyal\n(37-60m)', 3: 'Champion\n(61+m)'}
tb_churn = df.groupby('tenure_band')['churn'].apply(lambda x: (x=='Yes').mean())
axes[1].bar([tb_labels[i] for i in tb_churn.index], tb_churn.values,
            color=['#e74c3c','#f39c12','#3498db','#2ecc71'], edgecolor='white')
axes[1].set_title('Churn Rate by Tenure Band', fontweight='bold')
axes[1].set_ylabel('Churn Rate')

# revenue_at_risk
rar_churn = df.groupby('revenue_at_risk')['churn'].apply(lambda x: (x=='Yes').mean())
axes[2].bar(['Standard', 'Revenue at Risk'], rar_churn.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[2].set_title('Churn Rate: Revenue at Risk Flag', fontweight='bold')
axes[2].set_ylabel('Churn Rate')
for i, val in enumerate(rar_churn.values):
    axes[2].text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Encode Binary Columns (Yes/No → 0/1)

Binary encoding is straightforward — no ordering issue.  
We also encode the target variable `churn` here.

In [ ]:
df = encode_binary(df)

# Also encode target
df['churn_label'] = (df['churn'] == 'Yes').astype(int)

print('Churn label distribution:')
print(df['churn_label'].value_counts())
print(f'\nChurn rate: {df["churn_label"].mean():.2%}')

## 4. Train / Test Split

**Important:** Split BEFORE any encoding or scaling.  
The test set must stay completely untouched until final evaluation.

In [ ]:
feature_cols = NUMERIC_COLS + NOMINAL_COLS

# Keep only columns that exist in our dataframe
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['churn_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Train churn rate: {y_train.mean():.2%}')
print(f'Test churn rate : {y_test.mean():.2%}  ← stratified, so same distribution')

## 5. Preprocessing — Scale + Encode

**Why fit on train only?**  
If we fit the scaler on the full dataset, the test set statistics (mean, std) leak into training.  
The model would perform better in evaluation than it would in real production.

```
WRONG  → preprocessor.fit_transform(X_all)  # test data influences the scaler
CORRECT → preprocessor.fit(X_train)          # scaler learns from train only
           preprocessor.transform(X_test)    # applies same params to test
```

In [ ]:
preprocessor = build_preprocessor()

# Fit on train, transform both — this is the correct pattern
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'Processed train shape: {X_train_proc.shape}')
print(f'Processed test shape : {X_test_proc.shape}')
print(f'\n(OneHotEncoder expanded {len(NOMINAL_COLS)} nominal columns into multiple binary columns)')

## 6. Handle Class Imbalance with SMOTE

Our dataset has ~26% churn rate — imbalanced.  
A naive model that predicts "No churn" for everyone gets 74% accuracy — but it's useless.

**SMOTE** (Synthetic Minority Oversampling Technique) creates synthetic churn examples  
so the model sees equal numbers of churned and retained customers during training.

**Apply SMOTE only on the training set** — never on test.

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train)

print('Before SMOTE:')
print(f'  Retained: {(y_train == 0).sum():,}  |  Churned: {(y_train == 1).sum():,}')
print('\nAfter SMOTE:')
print(f'  Retained: {(y_train_bal == 0).sum():,}  |  Churned: {(y_train_bal == 1).sum():,}')
print(f'\nNew train size: {X_train_bal.shape[0]:,} rows')

## 7. Save Processed Data

Save the processed arrays so the modeling notebook can load them directly  
without re-running all the preprocessing steps.

In [ ]:
import numpy as np
from pathlib import Path
import joblib

out = Path('../data/processed')
out.mkdir(parents=True, exist_ok=True)

np.save(out / 'X_train.npy', X_train_bal)
np.save(out / 'X_test.npy',  X_test_proc)
np.save(out / 'y_train.npy', y_train_bal.values)
np.save(out / 'y_test.npy',  y_test.values)

# Save preprocessor so we can apply it to new data in production
joblib.dump(preprocessor, out / 'preprocessor.joblib')

# Save feature names for SHAP plots later
feature_names = (
    NUMERIC_COLS +
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(NOMINAL_COLS))
)
pd.Series(feature_names).to_csv(out / 'feature_names.csv', index=False)

print('Saved to data/processed/:')
for f in sorted(out.iterdir()):
    print(f'  {f.name}')